<a href="https://colab.research.google.com/github/karkhutmaria/KarkhutM_NLP_hw/blob/main/%D0%9A%D0%B0%D1%80%D1%85%D1%83%D1%82_%D0%9C_%2C_%D0%BB%D0%B0%D0%B1%D1%80%D0%B0%D0%B12%2C_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

В этом практикуме мы рассмотрим работу с библиотекой **Gensim** для работы с векторными представлениями текста

Мы рассмотрим
- **Word2Vec** - векторные представления слов
- **FastText** - улучшенные представления с учетом морфологии  
- **Doc2Vec** - векторные представления документов


In [2]:
!pip install gensim

import gensim
import gensim.downloader as api
from gensim.models import Word2Vec, FastText, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 63.5 MB/s eta 0:00:00


## Часть 1: Word2Vec

### Что такое Word2Vec?

Word2Vec преобразует слова в векторы чисел так, что семантически похожие слова оказываются близко в векторном пространстве.

**Два основных алгоритма:**
- **CBOW** - предсказывает слово по контексту
- **Skip-gram** - предсказывает контекст по слову

**Загрузка предобученной модели**

In [3]:
w2v_model = api.load('glove-wiki-gigaword-100')

print(f"Размер словаря: {len(w2v_model.key_to_index)}")
print(f"Размерность векторов: {w2v_model.vector_size}")

[==================================================] 100.0% 128.1/128.1MB downloaded
Размер словаря: 400000
Размерность векторов: 100


Найдите документацию `gensim`: какие датасеты кроме `glove-wiki-gigaword-100` доступны в библиотеке?

Выберите 3 датасета и кратко опишите их (источник данных, примерный объем, зачем такой датасет может использоваться)

Прочие датасеты, доступные в библиотеке gensim:
1. `word2vec-google-news-300` $-$ это датасет всех текстов Google News, в нем содержится около 100 млрд. слов (в данной выборке только 3 миллиона). Может быть использован для задач корпусной лингвистики, поиска синонимов и схожих слов, создания обширного словаря публицистических текстов, разметки таких типов текстов, а также для задач NLP по генерации публицистических текстов;
2. `fake-news` $-$ корпус заведомо ложных новостей, состоящий из 12999 текстов. Подобный датасет можно использовать для того, чтобы определить наиболее часто используемые в таком роде новостей слова и выражения, а в сочетании с проверенными и достверными новостями можно даже попробовать обучить модель определять ложные и истинные новости;
3. `text8` $-$ собран из первых 100 млн. байтов текстовой информации на Википедии. Используется в тренировочных целях для задач корпусной лингвистики и NLP: поиск синонимов и схожих слов, создание разнообразного словаря текстов, разметка и возможно даже генерация текстов.

**Базовые операции с векторами**

In [4]:
# Получаем вектор слова
vector = w2v_model['computer']
print(f"Вектор слова 'computer': {vector[:5]}...")  # Показываем первые 5 чисел

# Вычисляем схожесть между словами
similarity = w2v_model.similarity('computer', 'laptop')
print(f"Схожесть 'computer' и 'laptop': {similarity:.4f}")

Вектор слова 'computer': [-0.16298   0.30141   0.57978   0.066548  0.45835 ]...
Схожесть 'computer' и 'laptop': 0.7024


**Поиск похожих слов**

In [5]:
# Находим похожие слова
similar_words = w2v_model.most_similar('python', topn=5)
print("Слова, похожие на 'python':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

Слова, похожие на 'python':
  monty: 0.6886
  php: 0.5865
  perl: 0.5784
  cleese: 0.5447
  flipper: 0.5113


**Задание**

1. Загрузите любой датасет из gensim на ваш выбор

In [6]:
w2v_model = api.load('word2vec-google-news-300')

print(f"Размер словаря: {len(w2v_model.key_to_index)}")
print(f"Размерность векторов: {w2v_model.vector_size}")

[==================================================] 100.0% 1662.8/1662.8MB downloaded
Размер словаря: 3000000
Размерность векторов: 300


2. Напишите функцию, которая принимает на вход любое слово и вовращает 10 наиболее близких по вектору слов

In [7]:
def similar(word):
  sim10 = w2v_model.most_similar(word, topn=10)
  print(f"Слова, похожие на {word}:")
  for w, score in sim10:
    print(f"{w}: {score:.2f}")

test_word = "apple"
test = similar(test_word)

Слова, похожие на apple:
apples: 0.72
pear: 0.65
fruit: 0.64
berry: 0.63
pears: 0.61
strawberry: 0.61
peach: 0.60
potato: 0.60
grape: 0.59
blueberry: 0.59


3. Обучите модель Word2Vec на тестовом датасете из ячейки ниже

Примените следующие настройки:

- размер вектора: 50
- размер окна: 3
- минимальная частота слова: 1
- потоков: 2
- использовать skip-gram

In [8]:
cooking_sentences = [
    ['варить', 'суп', 'овощи', 'морковь', 'картофель'],
    ['жарить', 'курица', 'сковорода', 'масло', 'специи'],
    ['печь', 'хлеб', 'мука', 'дрожжи', 'духовка'],
    ['резать', 'овощи', 'салат', 'помидоры', 'огурцы'],
    ['смешивать', 'ингредиенты', 'тесто', 'яйца', 'молоко'],
    ['варить', 'паста', 'вода', 'соль', 'соус'],
    ['гриль', 'мясо', 'овощи', 'уголь', 'барбекю'],
    ['тушить', 'говядина', 'горшок', 'вино', 'травы'],
    ['запекать', 'рыба', 'лимон', 'духовка', 'фольга'],
    ['готовить', 'завтрак', 'яичница', 'бекон', 'тост'],
    ['месить', 'тесто', 'пирог', 'начинка', 'яблоки'],
    ['кипятить', 'вода', 'чай', 'кофе', 'чашка'],
    ['мариновать', 'мясо', 'соус', 'специи', 'холодильник'],
    ['взбивать', 'сливки', 'сахар', 'десерт', 'торт'],
    ['парить', 'овощи', 'здоровое', 'питание', 'брокколи']
]

In [9]:
w2v_model = Word2Vec(sentences=cooking_sentences, vector_size=50, window=3, min_count=1, workers=2, sg=1)

In [10]:
print(f"Слова в словаре: {list(w2v_model.wv.key_to_index.keys())[:10]}...")

Слова в словаре: ['овощи', 'мясо', 'соус', 'вода', 'тесто', 'духовка', 'специи', 'варить', 'брокколи', 'питание']...


4. Проверьте модель

In [11]:
# Проверяем похожие слова в кулинарной тематике
try:
    similar = w2v_model.wv.most_similar('варить', topn=5)
    print("Слова, похожие на 'варить':\n")
    for word, score in similar:
        print(f"{word}: {score:.4f}")
except KeyError:
    print("Слово 'варить' не найдено в словаре")

Слова, похожие на 'варить':

вино: 0.2398
ингредиенты: 0.2172
хлеб: 0.1938
брокколи: 0.1846
кипятить: 0.1711


In [12]:
# Найдите слова, похожие на "духовка"
# Найдите слова, похожие на "овощи"

def sim5(word):
  try:
    sim5 = w2v_model.wv.most_similar(word, topn=5)
    print(f"Слова, похожие на '{word}':\n")
    for w, score in sim5:
      print(f"{w}: {score:.2f}")
  except KeyError:
    print(f"'{word}' отсутствует в словаре")

test1 = sim5("духовка")
print("\n")
test2 = sim5("овощи")

Слова, похожие на 'духовка':

ингредиенты: 0.32
десерт: 0.31
холодильник: 0.27
питание: 0.22
пирог: 0.21


Слова, похожие на 'овощи':

мариновать: 0.27
хлеб: 0.27
гриль: 0.25
фольга: 0.24
сахар: 0.21


## Часть 2: FastText

FastText улучшает Word2Vec, рассматривая слова как наборы символов (n-грамм). Это позволяет работать с редкими словами и опечатками

5. Обучите FastText на корпусе текстов из пункта 3. Используйте код ниже

In [13]:
ft_model = FastText(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2
)

6. Найдите слова, похожие на "варить", "духовка" и "овощи" с помощью обученной модели. Используйте код из пункта 4

In [14]:
def ft_sim5(word):
  try:
    sim5 = ft_model.wv.most_similar(word, topn=5)
    print(f"Слова, похожие на '{word}':\n")
    for w, score in sim5:
      print(f"{w}: {score:.2f}")
  except KeyError:
    print(f"'{word}' отсутствует в словаре")

word1 = ft_sim5("варить")
print("\n")
word2 = ft_sim5("духовка")
print("\n")
word3 = ft_sim5("овощи")

Слова, похожие на 'варить':

жарить: 0.54
парить: 0.48
месить: 0.35
тушить: 0.34
специи: 0.26


Слова, похожие на 'духовка':

взбивать: 0.46
лимон: 0.36
салат: 0.30
курица: 0.30
тост: 0.29


Слова, похожие на 'овощи':

жарить: 0.30
фольга: 0.26
морковь: 0.23
соус: 0.22
торт: 0.21


7. Сравните модели

Дана функция для сравнения Word2Vec и FastText

Придумайте 3 слова с опечатками и проверьте, найдет ли их FastText и Word2Vec

In [15]:
def compare_models(word):
    """Сравнивает представления слова в разных моделях"""
    print(f"\nСравнение для слова: '{word}'")

    # Word2Vec
    try:
        w2v_similar = w2v_model.wv.most_similar(word, topn=2)
        print(f"  Word2Vec: {[w for w, _ in w2v_similar]}")
    except KeyError:
        print(f"  Word2Vec: слово не найдено")

    # FastText
    try:
        ft_similar = ft_model.wv.most_similar(word, topn=2)
        print(f"  FastText: {[w for w, _ in ft_similar]}")
    except KeyError:
        print(f"  FastText: слово не найдено")

# Сравниваем для разных слов
compare_models('сол')
compare_models('дисерт')
compare_models('мясить')


Сравнение для слова: 'сол'
  Word2Vec: слово не найдено
  FastText: ['мясо', 'соль']

Сравнение для слова: 'дисерт'
  Word2Vec: слово не найдено
  FastText: ['картофель', 'горшок']

Сравнение для слова: 'мясить'
  Word2Vec: слово не найдено
  FastText: ['мясо', 'питание']


## Часть 3: Doc2Vec

Doc2Vec расширяет Word2Vec для создания векторных представлений целых документов (предложений, абзацев, статей)

In [16]:
# Создаем размеченные документы
documents = [
    "machine learning is interesting",
    "deep learning uses neural networks",
    "python programming for data science",
    "artificial intelligence is amazing",
    "computer vision processes images"
]

# Преобразуем в формат TaggedDocument
tagged_docs = []
for i, doc in enumerate(documents):
    tokens = doc.split()
    tagged_doc = TaggedDocument(words=tokens, tags=[f"doc_{i}"])
    tagged_docs.append(tagged_doc)

print("Размеченные документы:")
for doc in tagged_docs[:3]:
    print(f"  Слова: {doc.words}")
    print(f"  Тег: {doc.tags}")

Размеченные документы:
  Слова: ['machine', 'learning', 'is', 'interesting']
  Тег: ['doc_0']
  Слова: ['deep', 'learning', 'uses', 'neural', 'networks']
  Тег: ['doc_1']
  Слова: ['python', 'programming', 'for', 'data', 'science']
  Тег: ['doc_2']


In [17]:
# Обучаем Doc2Vec
doc_model = Doc2Vec(
    documents=tagged_docs,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    epochs=20
)

print("Doc2Vec модель обучена!")
print(f"Количество документов: {len(doc_model.dv.key_to_index)}")

Doc2Vec модель обучена!
Количество документов: 5


In [18]:
# Получаем вектор документа
doc_vector = doc_model.dv["doc_0"]
print(f"Вектор документа doc_0: {doc_vector[:5]}...")

# Находим похожие документы
similar_docs = doc_model.dv.most_similar("doc_0", topn=2)
print("\nДокументы, похожие на doc_0:")
for doc_tag, similarity in similar_docs:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {documents[doc_id]}")

Вектор документа doc_0: [-0.01057    -0.01198188 -0.01982618  0.01710627  0.00710373]...

Документы, похожие на doc_0:
  doc_1: 0.2735
    Текст: deep learning uses neural networks
  doc_2: 0.1275
    Текст: python programming for data science


In [19]:
# Сравниваем схожесть документов
def compare_documents(doc1_id, doc2_id):
    similarity = doc_model.dv.similarity(f"doc_{doc1_id}", f"doc_{doc2_id}")
    print(f"Схожесть doc_{doc1_id} и doc_{doc2_id}: {similarity:.4f}")
    print(f"  doc_{doc1_id}: {documents[doc1_id]}")
    print(f"  doc_{doc2_id}: {documents[doc2_id]}")

compare_documents(0, 1)  # machine learning vs deep learning
compare_documents(0, 3)  # machine learning vs AI

Схожесть doc_0 и doc_1: 0.2735
  doc_0: machine learning is interesting
  doc_1: deep learning uses neural networks
Схожесть doc_0 и doc_3: -0.0822
  doc_0: machine learning is interesting
  doc_3: artificial intelligence is amazing


8. Сравните схожесть doc_2 и doc_4

In [20]:
compare_documents(2, 4)

Схожесть doc_2 и doc_4: -0.0362
  doc_2: python programming for data science
  doc_4: computer vision processes images


9. Найдите самый похожий документ на doc_1

In [21]:
similar_doc = doc_model.dv.most_similar("doc_1", topn=1)
print("Самый похожий на doc_1 документ:")
for doc_tag, similarity in similar_doc:
    doc_id = int(doc_tag.split('_')[1])
    print(f"{doc_tag}: {similarity:.2f}")
    print(f"Текст: {documents[doc_id]}")

Самый похожий на doc_1 документ:
doc_0: 0.27
Текст: machine learning is interesting


10. Выберите любую из трёх моделей. Обучите модели с разной размерностью (10, 50, 100). Продемонстрируйте качество их работы на примере поиска похожих слов (выберите любые 3 примера, соответствующих тематике корпуса из пункта 4)

In [22]:
ft_model_10 = FastText(
    sentences=cooking_sentences,
    vector_size=10,
    window=3,
    min_count=1,
    workers=2
)

ft_model_50 = FastText(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2
)

ft_model_100 = FastText(
    sentences=cooking_sentences,
    vector_size=100,
    window=3,
    min_count=1,
    workers=2
)

In [26]:
def compare_models_sizedif(word):
    print(f"\nСравнение для слова: '{word}'")

    try:
        similar1 = ft_model_10.wv.most_similar(word, topn=5)
        print(f"FastText (vector = 10): {[w for w, _ in similar1]}")
    except KeyError:
        print(f"FastText (vector = 10): слово не найдено")

    try:
        similar2 = ft_model_50.wv.most_similar(word, topn=5)
        print(f"FastText (vector = 50): {[w for w, _ in similar2]}")
    except KeyError:
        print(f"FastText (vector = 50): слово не найдено")

    try:
        similar3 = ft_model_100.wv.most_similar(word, topn=5)
        print(f"FastText (vector = 100): {[w for w, _ in similar3]}")
    except KeyError:
        print(f"FastText (vector = 100): слово не найдено")

compare_models_sizedif("барбекю")
print("\n")
compare_models_sizedif("травы")
print("\n")
compare_models_sizedif("сливки")


Сравнение для слова: 'барбекю'
FastText (vector = 10): ['пирог', 'тушить', 'специи', 'горшок', 'хлеб']
FastText (vector = 50): ['рыба', 'мясо', 'соль', 'парить', 'тушить']
FastText (vector = 100): ['рыба', 'ингредиенты', 'яйца', 'смешивать', 'мука']



Сравнение для слова: 'травы'
FastText (vector = 10): ['чашка', 'мука', 'варить', 'печь', 'масло']
FastText (vector = 50): ['завтрак', 'рыба', 'морковь', 'уголь', 'мясо']
FastText (vector = 100): ['торт', 'помидоры', 'мука', 'сливки', 'резать']



Сравнение для слова: 'сливки'
FastText (vector = 10): ['месить', 'запекать', 'уголь', 'здоровое', 'сковорода']
FastText (vector = 50): ['соус', 'сахар', 'тушить', 'питание', 'фольга']
FastText (vector = 100): ['начинка', 'резать', 'вода', 'варить', 'фольга']


Стоит отметить, что чем длиннее вектор, тем лучше и точнее модель ищет схожие слова. Во всех сравнениях с моделью, размер вектора которой 10, она справляется значительно хуже двух других моделей.